# HPO smoke test — Colab and CPU

This lightweight notebook validates the additions without duplicating framework code. It uses tiny synthetic 32×32 data, the repository model builders and training loop, and bounded studies. GPU is used when available; CPU is a supported fallback.

Coverage keywords: Python dictionary, CSV, Proxy, Successive halving, Full, Continuous, Resume, Pareto.

## 1. Setup and dependency checks

In [1]:
from pathlib import Path
import os, sys, subprocess, json, shutil

REPO_URL = "https://github.com/TrueRottweiler/WashingtonCsed504.git"
BRANCH = "feature/hpo-framework"  # change to the branch containing these additions
REPO_ROOT = Path("/content/WashingtonCsed504")
USE_EXISTING_CHECKOUT = (REPO_ROOT / "src/a1-cv/hpo").exists()

if not USE_EXISTING_CHECKOUT:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)], check=True)

CV_DIR = REPO_ROOT / "src/a1-cv"
os.chdir(CV_DIR)
if str(CV_DIR) not in sys.path:
    sys.path.insert(0, str(CV_DIR))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "hpo_requirements.txt"], check=True)
import torch, optuna, hpo
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Optuna:", optuna.__version__)
print("HPO package:", hpo.__file__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
Optuna: 4.9.0
HPO package: /content/WashingtonCsed504/src/a1-cv/hpo/__init__.py


## 2. Hardware detection and resource plan

In [2]:
from hpo.hardware import detect_hardware
from hpo.scheduler import plan_resources
hardware = detect_hardware()
plan = plan_resources(hardware, device="auto", requested_concurrency=1)
print(json.dumps(hardware.to_dict(), indent=2, default=str))
print(json.dumps(plan.to_dict(), indent=2, default=str))

{
  "operating_system": "Linux 6.6.122+",
  "notebook": true,
  "colab": true,
  "python_version": "3.12.13",
  "torch_version": "2.11.0+cu128",
  "cuda_available": true,
  "cuda_version": "12.8",
  "cudnn_version": 91900,
  "gpu_count": 1,
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "total_vram_gb": 14.56317138671875,
      "available_vram_gb": 14.5634765625,
      "compute_capability": [
        7,
        5
      ]
    }
  ],
  "fp16_supported": true,
  "bf16_supported": true,
  "bf16_native": false,
  "tf32_supported": false,
  "mps_available": false,
  "physical_cpu_cores": 1,
  "logical_cpu_threads": 2,
  "available_ram_gb": 11.400680541992188,
  "total_ram_gb": 12.671417236328125,
  "storage_free_gb": 65.86295700073242,
  "torch_compile_available": true,
  "multiprocessing_start_method": null,
  "nvidia_smi": {
    "gpus": [
      {
        "index": 0,
        "name": "Tesla T4",
        "memory_total_mb": 15360.0,
        "memory_free_mb": 14913.0,
     

## 3. Search-space inputs: Python dictionary, list, CSV, and manual input

In [3]:
from hpo.search_space import normalize_space, load_csv, combination_count, preview_rows
from hpo.notebook_api import preview_dataframe, optional_widgets

DICT_SPACE = {
    "optimizer": {"type": "categorical", "choices": ["sgd", "adamw"], "default": "sgd"},
    "learning_rate": {"type": "float", "low": 1e-4, "high": 0.2, "log": True, "default": 0.01},
    "batch_size": {"type": "categorical", "choices": [16, 32], "default": 16},
    "momentum": {"type": "float", "low": 0.8, "high": 0.95, "step": 0.05, "default": 0.9, "condition": 'optimizer == "sgd"'},
    "beta1": {"type": "float", "low": 0.8, "high": 0.95, "step": 0.05, "default": 0.9, "condition": 'optimizer == "adamw"'},
}
LIST_SPACE = [{"name": name, **spec} for name, spec in DICT_SPACE.items()]
MANUAL_SPACE = DICT_SPACE.copy()  # edit this normal Python cell
CSV_PATH = CV_DIR / "hpo_configs/search_spaces/resnet18_cifar.csv"

spaces = {
    "dictionary": normalize_space(DICT_SPACE, source_name="notebook dictionary"),
    "list": normalize_space(LIST_SPACE, source_name="notebook list"),
    "manual": normalize_space(MANUAL_SPACE, source_name="manual notebook input"),
    "csv": load_csv(CSV_PATH),
}
for name, specs in spaces.items():
    print(name, "finite combinations:", combination_count(specs))
    display(preview_dataframe(specs))

widgets = optional_widgets(DICT_SPACE)
if widgets is not None:
    display(widgets)

dictionary finite combinations: None


,name,type,low,high,choices,step,log,default,condition,enabled,source,item,description
0,optimizer,categorical,NaN,NaN,"[sgd, adamw]",NaN,False,sgd,None,True,notebook dictionary,1,None
1,learning_rate,float,0.0001,0.20,[],NaN,True,0.01,None,True,notebook dictionary,2,None
2,batch_size,categorical,NaN,NaN,"[16, 32]",NaN,False,16,None,True,notebook dictionary,3,None
3,momentum,float,0.8000,0.95,[],0.05,False,0.9,"optimizer == ""sgd""",True,notebook dictionary,4,None
4,beta1,float,0.8000,0.95,[],0.05,False,0.9,"optimizer == ""adamw""",True,notebook dictionary,5,None


list finite combinations: None


,name,type,low,high,choices,step,log,default,condition,enabled,source,item,description
0,optimizer,categorical,NaN,NaN,"[sgd, adamw]",NaN,False,sgd,None,True,notebook list,1,None
1,learning_rate,float,0.0001,0.20,[],NaN,True,0.01,None,True,notebook list,2,None
2,batch_size,categorical,NaN,NaN,"[16, 32]",NaN,False,16,None,True,notebook list,3,None
3,momentum,float,0.8000,0.95,[],0.05,False,0.9,"optimizer == ""sgd""",True,notebook list,4,None
4,beta1,float,0.8000,0.95,[],0.05,False,0.9,"optimizer == ""adamw""",True,notebook list,5,None


manual finite combinations: None


,name,type,low,high,choices,step,log,default,condition,enabled,source,item,description
0,optimizer,categorical,NaN,NaN,"[sgd, adamw]",NaN,False,sgd,None,True,manual notebook input,1,None
1,learning_rate,float,0.0001,0.20,[],NaN,True,0.01,None,True,manual notebook input,2,None
2,batch_size,categorical,NaN,NaN,"[16, 32]",NaN,False,16,None,True,manual notebook input,3,None
3,momentum,float,0.8000,0.95,[],0.05,False,0.9,"optimizer == ""sgd""",True,manual notebook input,4,None
4,beta1,float,0.8000,0.95,[],0.05,False,0.9,"optimizer == ""adamw""",True,manual notebook input,5,None


csv finite combinations: None


,name,type,low,high,choices,step,log,default,condition,enabled,source,item,description
0,optimizer,categorical,NaN,NaN,"[sgd, adamw]",NaN,False,sgd,None,True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,2,Optimizer family
1,learning_rate,float,0.000010,0.3000,[],NaN,True,0.1,None,True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,3,Base learning rate
2,batch_size,categorical,NaN,NaN,"[64, 128, 256, 512]",NaN,False,256,None,True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,4,Global batch size
3,weight_decay,float,0.000001,0.1000,[],NaN,True,0.0005,None,True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,5,Weight decay
4,momentum,float,0.800000,0.9900,[],0.0100,False,0.9,optimizer == 'sgd',True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,6,SGD momentum
5,nesterov,bool,NaN,NaN,"[False, True]",NaN,False,True,optimizer == 'sgd',True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,7,Nesterov acceleration
6,dampening,fixed,NaN,NaN,[],NaN,False,0.0,optimizer == 'sgd',True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,8,SGD dampening
7,beta1,float,0.800000,0.9900,[],0.0100,False,0.9,"optimizer in ['adam', 'adamw']",True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,9,Adam beta1
8,beta2,float,0.950000,0.9999,[],0.0001,False,0.999,"optimizer in ['adam', 'adamw']",True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,10,Adam beta2
9,epsilon,categorical,NaN,NaN,"[1e-08, 1e-07]",NaN,False,0.0,"optimizer in ['adam', 'adamw']",True,/content/WashingtonCsed504/src/a1-cv/hpo_confi...,11,Adam epsilon


## 4. Repository model and dataset adapters

In [4]:
import torch
from hpo.adapters import RepoModules, build_trial_model, build_trial_dataset
modules = RepoModules(REPO_ROOT)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bundle = build_trial_dataset(modules, {"name":"synthetic","train_examples":64,"validation_examples":32,"test_examples":32,"num_classes":10}, {"seed":42}, device)
resnet = build_trial_model(modules, {"name":"resnet18"}, bundle.num_classes, device)
vit = build_trial_model(modules, {"name":"vit","hidden_dim":48,"layers":1,"heads":3,"mlp_dim":96,"patch_size":8}, bundle.num_classes, device)
print("Dataset:", bundle.name, bundle.train_examples, bundle.validation_examples, bundle.test_examples)
print("ResNet output:", tuple(resnet(next(bundle.train.epoch(2, True))[0]).shape))
print("ViT output:", tuple(vit(next(bundle.train.epoch(2, True))[0]).shape))

Dataset: synthetic 64 32 32
ResNet output: (2, 10)
ViT output: (2, 10)


## 5. Batch and runtime calibration

In [5]:
from hpo.calibration import calibrate_batch_sizes
calibration = calibrate_batch_sizes(
    lambda: build_trial_model(modules, {"name":"vit","hidden_dim":48,"layers":1,"heads":3,"mlp_dim":96,"patch_size":8}, 10, device),
    lambda batch: (torch.randn(batch,3,32,32,device=device), torch.randint(0,10,(batch,),device=device)),
    device=device,
    candidates=[2,4,8],
    precision="fp16" if device.type == "cuda" else "fp32",
    warmup_steps=1,
    measure_steps=1,
)
print(json.dumps(calibration.to_dict(), indent=2))

{
  "measurements": [
    {
      "batch_size": 2,
      "status": "completed",
      "seconds_per_step": 0.004611960000033832,
      "examples_per_second": 433.65510541837494,
      "peak_allocated_mb": 60.8037109375,
      "peak_reserved_mb": 86.0,
      "error": null
    },
    {
      "batch_size": 4,
      "status": "completed",
      "seconds_per_step": 0.004782911999996031,
      "examples_per_second": 836.3105990667023,
      "peak_allocated_mb": 60.97607421875,
      "peak_reserved_mb": 86.0,
      "error": null
    },
    {
      "batch_size": 8,
      "status": "completed",
      "seconds_per_step": 0.00594931600005566,
      "examples_per_second": 1344.6923982395883,
      "peak_allocated_mb": 61.32470703125,
      "peak_reserved_mb": 86.0,
      "error": null
    }
  ],
  "largest_fitting_batch": 8,
  "highest_throughput_batch": 8,
  "recommended_candidates": [
    8
  ],
  "memory_headroom_fraction": 0.15
}


## 6. Tiny proxy, successive-halving, full, and continuous studies

In [6]:
required_names = ["REPO_ROOT", "CV_DIR", "device"]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Run Sections 1 through 5 first. "
        f"Missing variables: {missing_names}"
    )

print("Sections 1–5 are ready.")
print("Repository:", REPO_ROOT)
print("CV directory:", CV_DIR)
print("Device:", device)

Sections 1–5 are ready.
Repository: /content/WashingtonCsed504
CV directory: /content/WashingtonCsed504/src/a1-cv
Device: cuda


In [7]:
from pathlib import Path
import copy
import json
import shutil
import yaml

from hpo.config import load_study_config
from hpo.study import HpoStudy


# Fail with an actionable message instead of a NameError.
required_names = ["REPO_ROOT", "CV_DIR", "device"]
missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Run Sections 1 through 5 before Section 6. "
        f"Missing variables: {missing_names}"
    )


# Remove only earlier smoke-test outputs.
OUTPUT_ROOT = Path("/content/hpo_smoke_outputs")
shutil.rmtree(OUTPUT_ROOT, ignore_errors=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


# Load the repository smoke configuration.
BASE = yaml.safe_load(
    (
        CV_DIR
        / "hpo_configs/smoke/resnet18_cifar10.yaml"
    ).read_text()
)


# Use a tiny synthetic dataset and tiny ViT.
BASE["dataset"] = {
    "name": "synthetic",
    "train_examples": 64,
    "validation_examples": 32,
    "test_examples": 32,
    "num_classes": 10,
}

BASE["model"] = {
    "name": "vit",
    "hidden_dim": 48,
    "layers": 1,
    "heads": 3,
    "mlp_dim": 96,
    "patch_size": 8,
}

BASE["runtime"].update(
    {
        "device": device.type,
        "precision": (
            "fp16"
            if device.type == "cuda"
            else "fp32"
        ),
        "concurrent_trials": 1,
        "intraop_threads": 1,
        "interop_threads": 1,
    }
)


# IMPORTANT:
# Parameters belong directly under search_space.
# There must not be an extra "inline" wrapper.
BASE["search_space"] = {
    "optimizer": {
        "type": "fixed",
        "default": "adamw",
    },
    "learning_rate": {
        "type": "categorical",
        "choices": [0.001, 0.003],
        "default": 0.001,
    },
    "batch_size": {
        "type": "fixed",
        "default": 16,
    },
    "weight_decay": {
        "type": "fixed",
        "default": 0.01,
    },
}


BASE["proxy"].update(
    {
        "trials": 2,
        "promote_top_k": 1,
        "budget": {
            "epochs": 1,
            "max_steps": 1,
            "data_fraction": 1.0,
            "validation_fraction": 1.0,
        },
    }
)

BASE["successive_halving"].update(
    {
        "rung_budgets": [2],
        "minimum_trials_per_rung": 1,
    }
)

BASE["full"].update(
    {
        "trials": 1,
        "budget": {
            "epochs": 1,
            "max_steps": 1,
            "data_fraction": 1.0,
            "validation_fraction": 1.0,
            "seeds": [42],
        },
    }
)

BASE["continuous"].update(
    {
        "enabled": False,
        "maximum_trials": 2,
        "maximum_wall_time_hours": 0.05,
        "stop_after_no_improvement_trials": 2,
    }
)


def make_smoke_config(
    study_name: str,
    mode: str,
) -> dict:
    """Build one isolated smoke-study configuration."""

    raw = copy.deepcopy(BASE)

    raw["study"].update(
        {
            "name": study_name,
            "output_dir": str(OUTPUT_ROOT),
            "storage_path": str(
                OUTPUT_ROOT / f"{study_name}.db"
            ),
        }
    )

    # The parser gives search.mode priority over a top-level mode.
    raw.setdefault("search", {})["mode"] = mode

    return raw


# Validate the search-space structure before training.
preflight_raw = make_smoke_config(
    "smoke-preflight",
    "proxy",
)

preflight_config = load_study_config(
    preflight_raw
)

parameter_names = [
    spec.name
    for spec in preflight_config.search_space
]

print("Normalized parameters:", parameter_names)

assert parameter_names == [
    "optimizer",
    "learning_rate",
    "batch_size",
    "weight_decay",
]

assert preflight_config.mode == "proxy"

print("Search-space preflight: PASSED")


# Run the three explicit modes.
SUMMARIES = {}

for mode in [
    "proxy",
    "successive_halving",
    "full",
]:
    raw = make_smoke_config(
        f"smoke-{mode}",
        mode,
    )

    validated_config = load_study_config(raw)

    print(
        f"Starting {mode}: "
        f"validated mode={validated_config.mode}"
    )

    assert validated_config.mode == mode

    study = HpoStudy(
        validated_config,
        repo_root=REPO_ROOT,
    )

    SUMMARIES[mode] = study.run()


# Run bounded continuous proxy mode.
continuous_raw = make_smoke_config(
    "smoke-continuous",
    "proxy",
)

continuous_raw["continuous"]["enabled"] = True
continuous_raw["continuous"]["strategy"] = "proxy"

continuous_config = load_study_config(
    continuous_raw
)

assert continuous_config.mode == "proxy"
assert continuous_config.continuous.enabled

print(
    "Starting continuous: "
    f"base mode={continuous_config.mode}, "
    f"strategy={continuous_config.continuous.strategy}"
)

SUMMARIES["continuous"] = HpoStudy(
    continuous_config,
    repo_root=REPO_ROOT,
).run()


print(
    json.dumps(
        SUMMARIES,
        indent=2,
        default=str,
    )
)

Normalized parameters: ['optimizer', 'learning_rate', 'batch_size', 'weight_decay']
Search-space preflight: PASSED
Starting proxy: validated mode=proxy


[I 2026-07-21 06:43:20,966] A new study created in RDB with name: smoke-proxy


Starting successive_halving: validated mode=successive_halving


[I 2026-07-21 06:43:25,716] A new study created in RDB with name: smoke-successive_halving


Starting full: validated mode=full


[I 2026-07-21 06:43:28,533] A new study created in RDB with name: smoke-full


Starting continuous: base mode=proxy, strategy=proxy


[I 2026-07-21 06:43:29,454] A new study created in RDB with name: smoke-continuous


{
  "proxy": {
    "study": "smoke-proxy",
    "mode": "proxy",
    "study_dir": "/content/hpo_smoke_outputs/smoke-proxy",
    "storage_path": "/content/hpo_smoke_outputs/smoke-proxy.db",
    "records": 2,
    "status_counts": {
      "completed": 2
    },
    "candidate_status_counts": {
      "completed": 2
    },
    "report": {
      "records": 2,
      "completed_candidates": 2,
      "pareto_candidates": 1,
      "plots": {
        "accuracy_vs_time": "/content/hpo_smoke_outputs/smoke-proxy/accuracy_vs_time.png",
        "accuracy_vs_memory": "/content/hpo_smoke_outputs/smoke-proxy/accuracy_vs_memory.png",
        "optimization_history": "/content/hpo_smoke_outputs/smoke-proxy/optimization_history.png"
      },
      "status_counts": {
        "completed": 2
      }
    },
    "optuna_analysis": {
      "optuna_trials_csv": "/content/hpo_smoke_outputs/smoke-proxy/optuna_trials.csv",
      "parameter_importance_status": "skipped: at least 5 completed Optuna trials are required",
 

## 7. Persistence, resumption, pruning, Pareto, and export

In [8]:
from hpo.persistence import read_jsonl
from hpo.reporting import export_reports
from hpo.config import load_study_config
from hpo.schemas import ObjectiveSpec

sh_dir=OUTPUT_ROOT/"smoke-successive_halving"
events=read_jsonl(sh_dir/"events.jsonl")
assert any(e.get("event")=="candidate_pruned" for e in events), "halving did not record pruning"
# Reopen the completed study; completed work must not be duplicated.
raw = copy.deepcopy(BASE)

raw["study"].update(
    {
        "name": "smoke-successive_halving",
        "output_dir": str(OUTPUT_ROOT),
        "storage_path": str(
            OUTPUT_ROOT
            / "smoke-successive_halving.db"
        ),
    }
)

raw.setdefault(
    "search",
    {},
)["mode"] = "successive_halving"
before=len(read_jsonl(sh_dir/"trials.jsonl")); resumed=HpoStudy(raw, repo_root=REPO_ROOT).run(); after=len(read_jsonl(sh_dir/"trials.jsonl"))
assert before==after
report=export_reports(sh_dir,[ObjectiveSpec("validation_top1","maximize",True),ObjectiveSpec("wall_seconds","minimize",False)])
print("Resume summary:", resumed)
print("Report:", report)
print("Exports:", sorted(p.name for p in sh_dir.iterdir()))

[I 2026-07-21 06:43:31,302] Using an existing study with name 'smoke-successive_halving' instead of creating a new one.


Resume summary: {'study': 'smoke-successive_halving', 'mode': 'successive_halving', 'study_dir': '/content/hpo_smoke_outputs/smoke-successive_halving', 'storage_path': '/content/hpo_smoke_outputs/smoke-successive_halving.db', 'records': 5, 'status_counts': {'completed': 4, 'seed_completed': 1}, 'candidate_status_counts': {'completed': 1, 'pruned': 1}, 'report': {'records': 5, 'completed_candidates': 2, 'pareto_candidates': 1, 'plots': {'accuracy_vs_time': '/content/hpo_smoke_outputs/smoke-successive_halving/accuracy_vs_time.png', 'accuracy_vs_memory': '/content/hpo_smoke_outputs/smoke-successive_halving/accuracy_vs_memory.png', 'optimization_history': '/content/hpo_smoke_outputs/smoke-successive_halving/optimization_history.png', 'proxy_vs_full': '/content/hpo_smoke_outputs/smoke-successive_halving/proxy_vs_full.png'}, 'status_counts': {'completed': 4, 'seed_completed': 1}}, 'optuna_analysis': {'optuna_trials_csv': '/content/hpo_smoke_outputs/smoke-successive_halving/optuna_trials.csv'

## 8. Final pass/fail summary

In [9]:
checks={
"imports": True,
"hardware": hardware is not None,
"list_input": bool(spaces["list"]),
"dictionary_input": bool(spaces["dictionary"]),
"csv_input": bool(spaces["csv"]),
"manual_input": bool(spaces["manual"]),
"model_adapter": resnet is not None and vit is not None,
"dataset_adapter": bundle.train_examples==64,
"calibration": calibration.largest_fitting_batch is not None,
"proxy": "proxy" in SUMMARIES,
"successive_halving": "successive_halving" in SUMMARIES,
"full": "full" in SUMMARIES,
"continuous": "continuous" in SUMMARIES,
"pruning": any(e.get("event")=="candidate_pruned" for e in events),
"resume_no_duplicates": before==after,
"pareto_export": (sh_dir/"pareto_trials.csv").exists(),
}
print(json.dumps(checks, indent=2))
assert all(checks.values())
print("FINAL RESULT: PASS")

{
  "imports": true,
  "hardware": true,
  "list_input": true,
  "dictionary_input": true,
  "csv_input": true,
  "manual_input": true,
  "model_adapter": true,
  "dataset_adapter": true,
  "calibration": true,
  "proxy": true,
  "successive_halving": true,
  "full": true,
  "continuous": true,
  "pruning": true,
  "resume_no_duplicates": true,
  "pareto_export": true
}
FINAL RESULT: PASS
